# NLP Mastery Journey — Module 7: The Transformer Architecture, End to End

Module 6 ended with attention breaking RNNs' fixed-vector bottleneck, and one limitation still standing: recurrence forces sequential, non-parallel computation. This module removes recurrence **entirely** and builds the architecture that replaced it — the **Transformer**, from the 2017 paper *"Attention Is All You Need."* This is the architecture underneath BERT, GPT, T5, and Claude.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | Self-attention, from scratch | The core mechanism — every token looking at every other token, in parallel |
| 2 | Multi-head attention | Why one attention pattern isn't enough |
| 3 | Positional encoding | How position gets back in, once recurrence is gone |
| 4 | The full Transformer block | Residual connections, LayerNorm, feedforward sublayers |
| 5 | Encoder vs. decoder vs. encoder-decoder | BERT-style vs. GPT-style vs. T5-style, and when each is used |
| 6 | Building a mini Transformer classifier | End-to-end PyTorch, compared against Module 6's RNN/LSTM results |
| 7 | Subword tokenization (BPE) | Why Transformers don't tokenize on whitespace |
| 8 | From scratch to pretrained | Why virtually nobody trains a Transformer from scratch in practice |
| 9 | Production: inference costs, KV-caching, quantization | The engineering reality of serving these models at scale |

### How to use this notebook
- Parts 1–3 and 7 implement the actual math **in plain NumPy**, so every cell there runs immediately, with no installs — you'll see real attention weights and real positional encoding values, not just descriptions.
- Parts 4–6 use **PyTorch**, same as Module 6 — install once with the setup cell.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.


## 0. Setup

In [ ]:
# %pip install numpy matplotlib
# %pip install torch --index-url https://download.pytorch.org/whl/cpu
# %pip install transformers tokenizers   # for Part 7's real-tokenizer section and Part 8

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

print("Setup note: uncomment the pip install lines above the first time you run this.")


## Part 1 — Self-Attention, From Scratch (in NumPy)

**Self-attention** lets every token in a sequence look directly at every other token (including itself) and decide how much to "attend" to each one — all computed **in parallel**, unlike an RNN's step-by-step recurrence. Each token produces three vectors, via three separate learned weight matrices:

- **Query (Q)**: "what am I looking for?"
- **Key (K)**: "what do I contain, that others might look for?"
- **Value (V)**: "what do I actually offer, once someone attends to me?"

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$: every query dotted with every key — a full $n \times n$ matrix of "how relevant is token $j$ to token $i$"
- $\sqrt{d_k}$: scaling factor (square root of the key dimension) that keeps the dot products from growing too large and pushing softmax into tiny gradients — this is literally why it's called *"scaled"* dot-product attention
- softmax: turns each row of relevance scores into a probability distribution (sums to 1)
- multiplying by $V$: takes a weighted average of every token's Value vector, weighted by how much attention it received


In [ ]:
# ── A toy sentence, represented as random token embeddings ──────────────────
# In a real Transformer these come from a trained Embedding layer (Module 6);
# here we use random vectors purely to demonstrate the ATTENTION MATH clearly.

tokens = ["the", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 8   # embedding dimension — kept small so the math is easy to read/print

X = np.random.randn(seq_len, d_model)   # (seq_len, d_model) — one row per token
print("Input embeddings shape:", X.shape)


In [ ]:
# ── Step 1: project X into Query, Key, Value using learned weight matrices ──
d_k = d_model   # for this single-head demo we keep Q/K/V the same size as d_model;
                # multi-head attention (Part 2) is what actually SPLITS this up

W_query = np.random.randn(d_model, d_k) * 0.1
W_key = np.random.randn(d_model, d_k) * 0.1
W_value = np.random.randn(d_model, d_k) * 0.1

Q = X @ W_query   # (seq_len, d_k)
K = X @ W_key     # (seq_len, d_k)
V = X @ W_value   # (seq_len, d_k)

print("Q, K, V shapes:", Q.shape, K.shape, V.shape)


In [ ]:
# ── Step 2: scaled dot-product attention ─────────────────────────────────────

def softmax(x, axis=-1):
    # subtract the max for numerical stability — standard trick, prevents
    # overflow when exponentiating large numbers, doesn't change the result
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (seq_len, seq_len) — raw relevance scores
    attention_weights = softmax(scores, axis=-1)   # each ROW sums to 1
    output = attention_weights @ V       # (seq_len, d_k) — weighted average of Values
    return output, attention_weights

output, attention_weights = scaled_dot_product_attention(Q, K, V)
print("Output shape:", output.shape)               # one contextualized vector PER TOKEN
print("Attention weights shape:", attention_weights.shape)   # (seq_len, seq_len)
print("Each row sums to 1:", attention_weights.sum(axis=1).round(3))


In [ ]:
# ── Visualizing the attention matrix ─────────────────────────────────────────
plt.figure(figsize=(5, 5))
plt.imshow(attention_weights, cmap="Blues")
plt.xticks(range(seq_len), tokens)
plt.yticks(range(seq_len), tokens)
plt.xlabel("Attending TO (Key)")
plt.ylabel("Attending FROM (Query)")
plt.title("Self-Attention Weights (untrained, random weights)")
plt.colorbar()
plt.tight_layout()
plt.show()

# With RANDOM, untrained weights this pattern is meaningless noise — the whole
# point of TRAINING a Transformer is learning W_query/W_key/W_value such that
# this matrix lights up on genuinely relevant token pairs (e.g. a pronoun
# attending strongly to the noun it refers to).


## Part 2 — Multi-Head Attention

A single attention pattern can only capture one "kind" of relationship at a time. **Multi-head attention** runs several smaller attention computations **in parallel**, each with its own Q/K/V weight matrices, so different heads can specialize — one head might learn to track subject-verb agreement, another might track coreference (pronouns to the nouns they refer to), another might track local word order. The heads' outputs are concatenated and passed through one more linear layer to combine them.


In [ ]:
# ── Multi-head attention, from scratch ───────────────────────────────────────

def multi_head_attention(X, num_heads, d_model):
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
    d_k = d_model // num_heads   # each head gets a SLICE of the full dimension

    head_outputs = []
    all_head_weights = []
    for head in range(num_heads):
        # Each head gets ITS OWN randomly-initialized Q/K/V projections —
        # in a real model these are LEARNED, and end up different per head
        W_q = np.random.randn(d_model, d_k) * 0.1
        W_k = np.random.randn(d_model, d_k) * 0.1
        W_v = np.random.randn(d_model, d_k) * 0.1

        Q, K, V = X @ W_q, X @ W_k, X @ W_v
        head_output, head_weights = scaled_dot_product_attention(Q, K, V)
        head_outputs.append(head_output)
        all_head_weights.append(head_weights)

    concatenated = np.concatenate(head_outputs, axis=-1)   # (seq_len, d_model) again,
                                                             # since num_heads * d_k = d_model
    W_output = np.random.randn(d_model, d_model) * 0.1      # final linear layer that
                                                              # mixes information ACROSS heads
    final_output = concatenated @ W_output
    return final_output, all_head_weights

mha_output, head_weights_list = multi_head_attention(X, num_heads=4, d_model=d_model)
print("Multi-head output shape:", mha_output.shape)   # same shape as single-head output —
                                                        # multi-head is a richer computation,
                                                        # not a bigger output


In [ ]:
# ── Visualizing what DIFFERENT heads attend to ──────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, weights) in enumerate(zip(axes, head_weights_list)):
    ax.imshow(weights, cmap="Blues")
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
    ax.set_title(f"Head {i+1}")
plt.tight_layout()
plt.show()

# Again, these are untrained/random, so the patterns are noise here — but this
# is EXACTLY the kind of plot researchers use on a TRAINED model to discover
# what each head learned to specialize in.


In [ ]:
# 🔀 Choosing num_heads in practice
# | Model            | d_model | num_heads | d_k per head |
# |--------------------|-----------|-------------|-----------------|
# | BERT-base            | 768       | 12            | 64                |
# | BERT-large             | 1024      | 16            | 64                |
# | GPT-2 small              | 768       | 12            | 64                |
# Notice d_k (dimension PER head) tends to stay around 64 even as models grow —
# it's usually num_heads that scales up alongside d_model, not head size.

print("Typical head-count configurations shown above.")


## Part 3 — Positional Encoding

Self-attention alone has no idea about word **order** — `"dog bites man"` and `"man bites dog"` would produce identical attention computations, since attention just looks at content, not position (unlike an RNN, which processes tokens strictly in sequence and gets order "for free"). Transformers fix this by **adding a position-dependent vector to each token's embedding** before attention ever runs, so position information rides along with the content.

The original Transformer paper uses fixed **sinusoidal** encodings:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Different dimensions oscillate at different frequencies — this gives every position a unique "fingerprint" vector, and (bonus property) lets the model generalize to positions it never saw exact examples of during training, since the pattern is smooth and continuous.


In [ ]:
# ── Computing sinusoidal positional encodings ────────────────────────────────

def sinusoidal_positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]               # (seq_len, 1)
    dims = np.arange(d_model)[np.newaxis, :]                     # (1, d_model)

    angle_rates = 1 / (10000 ** (2 * (dims // 2) / d_model))      # frequency per dimension
    angles = positions * angle_rates                               # (seq_len, d_model)

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])   # even dimensions -> sine
    pe[:, 1::2] = np.cos(angles[:, 1::2])   # odd dimensions -> cosine
    return pe

pe = sinusoidal_positional_encoding(seq_len=50, d_model=64)
print("Positional encoding shape:", pe.shape)

plt.figure(figsize=(8, 5))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("Position in sequence")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

# Reading it: lower dimensions (top rows) oscillate FAST (change every position),
# higher dimensions (bottom rows) oscillate SLOWLY — together, every position
# gets a unique combination, like a fingerprint made of many clock hands
# spinning at different speeds.


In [ ]:
# In the actual model, this gets ADDED directly to the token embeddings:
# final_input = token_embeddings + positional_encoding[:seq_len]
# (elementwise addition, same shape on both sides — NOT concatenation)

# 🔀 Alternative: LEARNED positional embeddings
# Instead of a fixed sinusoidal formula, treat position like just another
# thing to embed: nn.Embedding(max_seq_len, d_model), trained end-to-end
# alongside everything else. BERT and GPT-2 both use LEARNED positional
# embeddings rather than the original paper's fixed sinusoidal version —
# simpler to implement, at the cost of not generalizing cleanly beyond the
# maximum sequence length seen during training (sinusoidal encodings can, in
# principle, extrapolate to unseen lengths; learned ones generally can't).

print("Positional encoding is ADDED to token embeddings — same shape, elementwise sum.")


## Part 4 — The Full Transformer Block

A real Transformer layer wraps attention in a few more essential pieces:

1. **Residual connections** (`x + Sublayer(x)`): lets gradients flow directly through the network via a shortcut path, which is what makes training very deep stacks of these blocks (12, 24, 96+ layers) actually feasible — without them, deep networks are notoriously hard to train.
2. **Layer Normalization**: normalizes each token's vector to stabilize training, applied either before (`Pre-LN`, the modern default) or after (`Post-LN`, the original paper's choice) each sublayer.
3. **Feedforward sublayer**: a small two-layer MLP applied independently to every position, adding extra representational capacity beyond what attention alone provides.

This needs **PyTorch** from here on — installed in the Setup cell.


In [ ]:
import torch
import torch.nn as nn

class TransformerEncoderBlock(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE — one full Transformer encoder layer, built from
    the pieces explained above. Stacking N of these (Part 6) IS a Transformer
    encoder — this is genuinely the whole architecture, just repeated.
    """
    def __init__(self, d_model, num_heads, d_ff=256, dropout=0.1):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, batch_first=True
        )
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        # ── Sublayer 1: self-attention, with a residual connection ──────────
        normed_x = self.norm1(x)                                   # Pre-LN: normalize BEFORE the sublayer
        attn_output, attn_weights = self.self_attention(
            normed_x, normed_x, normed_x,       # query, key, value are all the SAME
                                                 # tensor — this is what makes it SELF-attention
            key_padding_mask=attention_mask,    # tells attention to ignore padding tokens
        )
        x = x + self.dropout(attn_output)        # residual connection: add the ORIGINAL x back

        # ── Sublayer 2: feedforward, with a residual connection ─────────────
        normed_x = self.norm2(x)
        ff_output = self.feedforward(normed_x)
        x = x + self.dropout(ff_output)

        return x, attn_weights

# Quick shape check with dummy data:
dummy_input = torch.randn(2, 10, 32)   # (batch=2, seq_len=10, d_model=32)
block = TransformerEncoderBlock(d_model=32, num_heads=4)
output, weights = block(dummy_input)
print("Block output shape:", output.shape)     # SAME shape as input — this is important:
                                                # it's what lets you STACK many of these
print("Attention weights shape:", weights.shape)


In [ ]:
# 🔀 Pre-LN vs Post-LN (norm BEFORE vs AFTER each sublayer)
# | Variant   | Used by                         | Trade-off                                    |
# |------------|-------------------------------------|-------------------------------------------------|
# | Post-LN     | Original 2017 Transformer paper       | Slightly better final performance, but training |
# |              |                                        | is less stable, especially for very deep stacks   |
# | Pre-LN        | GPT-2/3, most modern LLMs                | Much more stable training, standard for large-scale models |

print("Pre-LN (used above) is the modern default for exactly this stability reason.")


## Part 5 — Encoder, Decoder, and Encoder-Decoder Architectures

The block from Part 4 is an **encoder** block — every token can attend to every OTHER token, in both directions, no restrictions. Three architecture families emerged from mixing encoder and decoder blocks differently:

| Architecture | Attention pattern | Famous examples | Best suited for |
|---------------|----------------------|---------------------|--------------------|
| **Encoder-only** | Full bidirectional (sees the whole input) | BERT, RoBERTa | Classification, NER, understanding tasks — NOT text generation |
| **Decoder-only** | **Causal** (each token can only attend to PAST tokens, not future ones) | GPT family, Claude, LLaMA | Text generation — the standard architecture for modern chat-style LLMs |
| **Encoder-decoder** | Encoder is bidirectional; decoder is causal AND cross-attends to the encoder's output | T5, the original Transformer (built for translation) | Sequence-to-sequence tasks: translation, summarization |

### Causal masking (the key difference for decoder-only models)
A decoder block is IDENTICAL to Part 4's encoder block, with one addition: a **mask** that forces each position's attention scores toward all FUTURE positions to $-\infty$ before the softmax — so after softmax, those positions get exactly 0 attention weight. This is what makes autoregressive generation (predicting the next token using only what came before) architecturally enforced, not just a training-time convention.


In [ ]:
# ── Causal masking, from scratch ──────────────────────────────────────────────

def causal_mask(seq_len):
    """
    📋 COPY-PASTE TEMPLATE
    Returns an upper-triangular mask of -inf above the diagonal, 0 elsewhere.
    Added to raw attention scores BEFORE softmax: -inf positions become
    exactly 0 after softmax (since exp(-inf) = 0), so they contribute nothing.
    """
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9   # k=1 excludes the diagonal
                                                                # itself (a token CAN attend to itself)
    return mask

mask = causal_mask(seq_len=6)
scores = Q @ K.T / np.sqrt(d_k)     # reusing Q, K from Part 1
masked_scores = scores + mask
causal_weights = softmax(masked_scores, axis=-1)

plt.figure(figsize=(5, 5))
plt.imshow(causal_weights, cmap="Blues")
plt.xticks(range(seq_len), tokens); plt.yticks(range(seq_len), tokens)
plt.title("Causal (masked) attention — note the empty upper-right triangle")
plt.colorbar()
plt.tight_layout()
plt.show()

# Notice: row "the" (position 0) can ONLY attend to itself. Row "mat"
# (the last position) can attend to everything — it's the only token that
# has seen the whole sentence. This exactly mirrors how GPT-style models
# generate text one token at a time, left to right.


## Part 6 — Building a Mini Transformer Classifier

Stack a few `TransformerEncoderBlock`s (Part 4) on top of an embedding + positional encoding, and you have a real (small) Transformer — the same shape of architecture as BERT, just with far fewer layers/dimensions. We reuse the exact sentiment dataset and vocabulary-building code from Module 6, so results are directly comparable to your RNN/LSTM numbers.


In [ ]:
class MiniTransformerClassifier(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE — a compact but architecturally faithful
    Transformer encoder for classification. Swap in more layers / a bigger
    d_model for a real project; the STRUCTURE below doesn't change.
    """
    def __init__(self, vocab_size, d_model=32, num_heads=4, num_layers=2,
                 d_ff=64, max_len=20, pad_idx=0):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)

        # A LEARNED positional embedding (like BERT/GPT-2) rather than
        # Part 3's fixed sinusoidal version — simpler to wire up in PyTorch:
        self.position_embedding = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand(batch_size, -1)

        x = self.token_embedding(token_ids) + self.position_embedding(positions)   # elementwise add,
                                                                                     # exactly as described in Part 3

        padding_mask = (token_ids == 0)   # True where a position is PADDING — tells
                                           # attention to ignore those positions entirely
        for layer in self.layers:
            x, _ = layer(x, attention_mask=padding_mask)

        x = self.final_norm(x)

        # For CLASSIFICATION, pool across the sequence into one vector.
        # Mean-pooling over the REAL (non-padding) tokens only:
        mask_expanded = (~padding_mask).unsqueeze(-1).float()   # 1.0 for real tokens, 0.0 for padding
        summed = (x * mask_expanded).sum(dim=1)
        counts = mask_expanded.sum(dim=1).clamp(min=1)           # avoid divide-by-zero
        pooled = summed / counts

        logits = self.classifier(pooled)
        return logits.squeeze(1)

# Quick shape check:
transformer_model = MiniTransformerClassifier(vocab_size=1000)   # placeholder vocab_size;
                                                                  # use len(vocab) from Module 6 in practice
dummy_ids = torch.randint(0, 1000, (4, 20))   # (batch=4, seq_len=20)
dummy_output = transformer_model(dummy_ids)
print("Output shape:", dummy_output.shape)   # (4,) — one logit per example, matches
                                              # Module 6's classifier outputs exactly


In [ ]:
# ── Training it — reusing Module 6's exact dataset-building + train_model() code ──
# (Not repeated here for brevity — copy hand_written_positive/negative,
# generate_reviews, tokenize/encode, SentimentDataset, and train_model
# straight from Module 6, then swap the model class:)
#
# transformer_model = MiniTransformerClassifier(vocab_size=len(vocab), max_len=MAX_LEN)
# transformer_history = train_model(transformer_model, train_loader, test_loader, epochs=15)
#
# print(f"Logistic Regression (Module 5): {logreg_accuracy:.3f}")
# print(f"LSTM (Module 6):                {lstm_history['test_accuracy'][-1]:.3f}")
# print(f"Mini Transformer:               {transformer_history['test_accuracy'][-1]:.3f}")
#
# 🔎 Another honest lesson, continuing Module 6's: Transformers are MORE
# data-hungry than RNNs, not less — they have no built-in sequential bias,
# so they need to LEARN that nearby tokens often matter more, purely from
# data. On our tiny dataset, don't be surprised if the mini Transformer
# doesn't beat the LSTM. Transformers' real advantage shows up at the scale
# of data and compute that lets you TRAIN THAT BIAS IN — which is also
# exactly why pretrained Transformers (Part 8) are almost always the
# practical choice over training one from scratch.

print("Full training comparison pattern shown above — reuses Module 6's training loop unchanged.")


## Part 7 — Subword Tokenization (BPE)

Modules 1–6 tokenized on whitespace. Real Transformers almost universally use **subword tokenization** instead — splitting rare/unknown words into smaller, reusable pieces. This solves two problems at once: a **fixed, manageable vocabulary size** (tens of thousands, not "every possible word"), and **graceful handling of unseen words** (an unfamiliar word decomposes into familiar pieces instead of becoming a single `<UNK>` token).

**Byte-Pair Encoding (BPE)**, the algorithm behind GPT's tokenizer, works by repeatedly merging the most frequent adjacent pair of symbols in the training corpus, building up a vocabulary from individual characters up to whole common words.


In [ ]:
# ── A minimal, from-scratch BPE implementation (the actual algorithm) ───────
# Real tokenizers (Part 7 continued) run this over billions of characters;
# here it's on a tiny corpus so you can watch every merge happen.

from collections import Counter, defaultdict

corpus = ["low", "lower", "lowest", "newest", "widest"]

def get_word_frequencies(corpus):
    # Represent each word as a tuple of CHARACTERS, with </w> marking word end
    # (so the tokenizer can tell "er" at a word boundary from "er" mid-word)
    return Counter(tuple(word) + ("</w>",) for word in corpus)

def get_pair_frequencies(word_freqs):
    pairs = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    bigram = "".join(pair)
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(bigram)   # merge the pair into one token
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs

word_freqs = get_word_frequencies(corpus)
num_merges = 10
merge_history = []

for step in range(num_merges):
    pairs = get_pair_frequencies(word_freqs)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)   # merge the MOST FREQUENT pair first
    word_freqs = merge_pair(best_pair, word_freqs)
    merge_history.append(best_pair)
    print(f"Merge {step+1}: {best_pair} (frequency {pairs[best_pair]})")

print("\nFinal tokenization of each word:")
for word in word_freqs:
    print(" ", word)


In [ ]:
# 🔀 BPE vs WordPiece vs SentencePiece — which tokenizer, which model
# | Algorithm       | Used by                      | Key difference                             |
# |--------------------|-----------------------------------|------------------------------------------------|
# | BPE                  | GPT-2/3/4, RoBERTa                   | Merges by raw frequency (what we implemented)    |
# | WordPiece              | BERT                                    | Merges by LIKELIHOOD improvement, not raw frequency|
# | SentencePiece            | T5, ALBERT, many multilingual models      | Treats input as a raw character stream (handles   |
# |                            |                                               | languages without whitespace, like Japanese, cleanly)|
#
# 📋 COPY-PASTE TEMPLATE — using a REAL, pretrained tokenizer in practice
# (you virtually never train BPE from scratch yourself — see Part 8):
#
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# tokens = tokenizer.tokenize("unbelievable")
# print(tokens)   # e.g. ['un', '##believable'] — "##" marks a subword
#                 # continuation (BERT's WordPiece convention)
# encoded = tokenizer("a sentence to encode", return_tensors="pt")
# print(encoded["input_ids"])   # ready to feed straight into a model

print("Real pretrained-tokenizer pattern shown above — needs internet the first run to download.")


## Part 8 — From Scratch to Pretrained: Why Nobody Trains a Transformer From Zero

Everything above is genuinely how BERT/GPT/Claude are built internally — but training one from scratch needs **massive** data (often hundreds of billions to trillions of tokens) and compute (weeks on large GPU/TPU clusters) that's simply out of reach for almost any individual project. The actual industry workflow at Google, Meta, and virtually every NLP team: start from a model **someone else already pretrained**, and either use it directly or **fine-tune** it on your specific task with a comparatively tiny amount of labeled data. This is exactly where Module 8 picks up.


In [ ]:
# ── Loading and using a pretrained Transformer (the real, everyday workflow) ─

# from transformers import AutoTokenizer, AutoModel
#
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# model = AutoModel.from_pretrained("bert-base-uncased")
#
# inputs = tokenizer("This movie was fantastic!", return_tensors="pt")
# outputs = model(**inputs)
#
# last_hidden_states = outputs.last_hidden_state   # (batch, seq_len, hidden_size) —
#                                                   # a contextual vector PER TOKEN,
#                                                   # exactly like Part 6's output, but
#                                                   # trained on a genuinely massive corpus
#
# # The [CLS] token's vector (position 0) is BERT's designated
# # "whole sentence summary" — commonly used for classification:
# sentence_vector = last_hidden_states[:, 0, :]

print("Pretrained-model loading pattern shown above — needs internet + the transformers library.")


## Part 9 — Production Reality: Inference Cost, KV-Caching, Quantization

Transformers are expensive to run, and production teams spend real engineering effort making them cheaper and faster.

### The core cost problem: attention is $O(n^2)$
Computing $QK^T$ for a sequence of length $n$ costs $O(n^2)$ time and memory — double the input length, and attention costs roughly 4x, not 2x. This is why context-length limits exist, and why a whole research area (sparse attention, sliding-window attention, linear attention approximations) exists purely to soften this scaling.

### KV-caching (the single biggest inference-speed trick for generation)
When a decoder-only model generates text one token at a time, recomputing every previous token's Key and Value vectors at every single new step would be wasteful — they never change once computed. **KV-caching** stores them once and reuses them, turning each new token's generation cost from "process the whole sequence again" into "process just this one new token." This is standard in every production LLM-serving system.

### Quantization
Storing weights in lower precision (e.g. 8-bit or 4-bit integers instead of 32-bit floats) shrinks memory footprint and speeds up inference substantially, at a small, usually-acceptable accuracy cost — the standard way large models get made cheap enough to run on more modest hardware.


In [ ]:
# 🔀 Techniques for taming Transformer inference cost, by what they target
# | Technique             | Targets                        | Typical use                                  |
# |----------------------------|------------------------------------|---------------------------------------------------|
# | KV-caching                    | Redundant recomputation during generation | Always on, in essentially every production LLM server |
# | Quantization (int8/int4)         | Memory footprint + raw compute            | Deploying large models on limited hardware          |
# | Sliding-window / sparse attention  | The O(n^2) attention cost itself           | Very long context windows                            |
# | Distillation                         | Overall model size                          | Creating a smaller, faster "student" model from a larger "teacher"|
# | Batching multiple requests             | GPU utilization                             | Serving many users concurrently, cost efficiency      |

print("Production inference-optimization landscape shown above.")


## Recap & What's Next

You implemented self-attention, multi-head attention, positional encoding, and causal masking **by hand in NumPy** — the actual math, not just a diagram — then built a real, working Transformer encoder block and classifier in PyTorch, trained BPE tokenization from scratch, and got a grounded picture of why production teams almost always start from a pretrained model rather than training from zero.

### Try this before the next lesson
1. Run the multi-head attention visualization (Part 2) again with `num_heads=1` vs `num_heads=8` and compare how the attention patterns differ.
2. Train the `MiniTransformerClassifier` from Part 6 on Module 6's dataset and add its accuracy to the Module 6 comparison table.
3. Run a real word (e.g. your own name, or a made-up word) through the from-scratch BPE tokenizer in Part 7 and see how it gets decomposed.

### Next lesson in your NLP mastery path
**Module 8: Fine-Tuning Pretrained Transformers** — taking a model like BERT or a small open GPT, adapting it to YOUR specific task (classification, NER, question answering) with a modest amount of labeled data, using Hugging Face's `transformers` library end to end — plus the fine-tuning vs. prompting vs. RAG decision that shapes how real LLM-based products (including ones built on Claude) actually get built today.
